# IBI Wave Reanalysis 2017–2024 — IceChunk + Monthly/Half-yearly Visualisation

**Dataset**: [IBI_ANALYSISFORECAST_WAV_005_005](https://data.marine.copernicus.eu/product/IBI_ANALYSISFORECAST_WAV_005_005/services)  
**Strategy**:
1. Stream hourly data lazily from Copernicus Marine year-by-year
2. Write real zarr chunks into an IceChunk store on `protocoast-data` S3
3. Re-open the store, resample to monthly / half-yearly averages
4. Visualise with hvplot (maps, time-series, seasonal climatology)

## 1. Imports & Configuration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import icechunk
from icechunk.xarray import to_icechunk
import copernicusmarine
import hvplot.xarray
import panel as pn
import cartopy.crs as ccrs
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=UserWarning)
_ = load_dotenv(f'{os.environ["HOME"]}/dotenv/protocoast.env', override=True)
pn.extension()

In [ ]:
BUCKET       = "rswain"
STORE_NAME   = "ibi-wave-2017-2024-v1"
ENDPOINT_URL = os.environ["ENDPOINT_URL"]
DATASET_ID   = "cmems_mod_ibi_wav_anfc_0.027deg_PT1H-i"   # multi-year hindcast
YEARS        = range(2023, 2024)    # the time range
CHUNKS       = {"time": 168, "latitude": 50, "longitude": 50}  # 168h = 1 week

print(f"Endpoint : {ENDPOINT_URL}")
print(f"Store    : s3://{BUCKET}/icechunk/{STORE_NAME}")

## 2. Explore Dataset (run once to confirm dataset ID & variables)

If the dataset ID below is wrong, adjust `DATASET_ID` in cell 1 and re-run.

In [ ]:
copernicusmarine.describe(dataset_id=DATASET_ID)

# Dask deployment

In [ ]:
#cluster_type = 'Local'    
#cluster_type = 'Coiled'
cluster_type = 'Coiled'
#cluster_type = 'Coiled'

In [ ]:
if cluster_type == 'Coiled':
    import coiled

    # Forward S3 + Copernicus Marine credentials to every worker
    worker_env = {k: os.environ[k] for k in [
        "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "ENDPOINT_URL",
        "COPERNICUSMARINE_SERVICE_USERNAME", "COPERNICUSMARINE_SERVICE_PASSWORD",
    ] if k in os.environ}

    cluster = coiled.Cluster(
        region="eu-central-1",
        arm=True,
        worker_vm_types=["t4g.large"],  # 4 GB — avoids OOM on large chunks
        worker_options={'nthreads': 2},
        n_workers=30,
        wait_for_workers=False,
        compute_purchase_option="spot_with_fallback",
        software='protocoast-notebook-arm',
        workspace='esip-lab',
        name="rswain-large",
        timeout=180,
        environ=worker_env,
    )

    client = cluster.get_client()

In [ ]:
client

## 3. IceChunk Storage Helper

In [ ]:
def make_storage_and_config():
    storage = icechunk.s3_storage(
        bucket=BUCKET,
        prefix=f"icechunk/{STORE_NAME}",
        from_env=True,
        endpoint_url=ENDPOINT_URL,
        region="not-used",
        force_path_style=True,
    )
    config = icechunk.RepositoryConfig.default()
    return storage, config

## 4. Write Hourly Data to IceChunk (year-by-year, idempotent)

Each year is fetched lazily from Copernicus Marine and written as real zarr chunks.  
Already-committed years are skipped automatically — safe to re-run.

> **Tip**: Test with `YEARS = range(2017, 2018)` first, then expand to the full range.

In [ ]:
storage = icechunk.s3_storage(
    bucket=BUCKET,
    prefix=f"icechunk/{STORE_NAME}",
    from_env=True,
    endpoint_url=ENDPOINT_URL,
    region="not-used",
    force_path_style=True,
)
config = icechunk.RepositoryConfig.default()

# open_or_create works on both a clean prefix and one with leftover objects
repo = icechunk.Repository.open_or_create(storage, config)

try:
    session = repo.readonly_session("main")
    ds_existing = xr.open_zarr(session.store, consolidated=False)
    written_years = set(pd.to_datetime(ds_existing.time.values).year)
    print(f"Existing store — years already written: {sorted(written_years)}")
except Exception:
    written_years = set()
    print("Empty store — will write from scratch.")

DATASET_START    = "2022-11-26"
DATASET_END      = "2024-12-31"
start_year       = pd.Timestamp(DATASET_START).year
end_year         = pd.Timestamp(DATASET_END).year
years_to_process = range(start_year, end_year + 1)
print(f"Years to process: {list(years_to_process)}")

first_write = not written_years

for year in years_to_process:
    if year in written_years:
        print(f"{year}: already in store, skipping")
        continue

    year_start = DATASET_START + "T00:00:00" if year == start_year else f"{year}-01-01T00:00:00"
    year_end   = f"{year}-12-31T23:00:00"

    print(f"{year}: fetching {year_start} → {year_end} ...")
    try:
        ds_year = copernicusmarine.open_dataset(
            dataset_id=DATASET_ID,
            start_datetime=year_start,
            end_datetime=year_end,
        )

        # Strip Zarr v2 codecs (numcodecs.Blosc) — IceChunk requires Zarr v3 codecs
        ds_year = ds_year.drop_encoding()

        print(f"  {year}: dims={dict(ds_year.sizes)}")

        session = repo.writable_session("main")
        if first_write:
            to_icechunk(ds_year, session, mode="w")
            first_write = False
        else:
            to_icechunk(ds_year, session, append_dim="time")

        commit_id = session.commit(f"Add IBI wave data: {year}")
        written_years.add(year)
        print(f"  {year}: committed — {commit_id}")

    except Exception as e:
        print(f"  {year}: FAILED — {e}")

print("\nAll done.")

## 5. Re-open Store & Verify

In [ ]:
storage = icechunk.s3_storage(
    bucket=BUCKET,
    prefix=f"icechunk/{STORE_NAME}",
    from_env=True,
    endpoint_url=ENDPOINT_URL,
    region="not-used",
    force_path_style=True,
)
config = icechunk.RepositoryConfig.default()

repo    = icechunk.Repository.open(storage, config)
session = repo.readonly_session("main")
# Open with monthly Dask chunks — avoids loading all time steps into RAM at once
ds      = xr.open_zarr(session.store, consolidated=False,
ds

In [ ]:
for snap in repo.ancestry(branch="main"):
    print(snap.written_at, snap.message)

## 6. Resample to Monthly & Half-yearly Averages

In [ ]:
# Keep lazy — hvplot pulls only what it needs per frame
ds_monthly = ds.resample(time="1ME").mean()
print(f"Monthly dataset: {len(ds_monthly.time)} time steps")
ds_monthly

In [ ]:
# Keep lazy — hvplot pulls only what it needs per frame
ds_6m = ds.resample(time="6ME").mean()
print(f"Half-yearly dataset: {len(ds_6m.time)} time steps")
ds_6m

## 7. Map — Animated Monthly Mean Significant Wave Height (VHM0)

In [ ]:
ds_monthly["VHM0"].hvplot(
    x="longitude", y="latitude",
    groupby="time",
    geo=True, tiles="OSM", project=True,
    crs=ccrs.PlateCarree(),
    cmap="viridis", clim=(0, 6),
    frame_width=600,
    title="Monthly mean significant wave height — VHM0 (m)",
)

## 8. Time-series at a Point — Monthly vs Half-yearly

In [ ]:
LON_PT, LAT_PT = -9.0, 43.5   # off NW Spain / Galician shelf

# Select point first, then compute — pulls ~26 and ~4 values, not the full grid
ts_monthly = ds_monthly["VHM0"].sel(longitude=LON_PT, latitude=LAT_PT, method="nearest").compute()
ts_6m      = ds_6m["VHM0"].sel(longitude=LON_PT, latitude=LAT_PT, method="nearest").compute()

(
    ts_monthly.hvplot.line(
        label="Monthly mean", color="steelblue",
        ylabel="VHM0 (m)", title=f"Significant wave height at lon={LON_PT}, lat={LAT_PT}",
        height=350, responsive=True,
    )
    * ts_6m.hvplot.scatter(
        label="6-monthly mean", color="orange", size=80,
    )
)

## 9. Multi-variable Map Dashboard (VHM0, VTPK, VMDR)

In [ ]:
var_titles = {
    "VHM0": "Significant wave height (m)",
    "VTPK": "Peak wave period (s)",
    "VMDR": "Mean wave direction (°)",
}

plots = []
for var, title in var_titles.items():
    if var not in ds_monthly:
        continue
    p = ds_monthly[var].hvplot(
        x="longitude", y="latitude",
        groupby="time",
        geo=True, tiles="OSM", project=True,
        crs=ccrs.PlateCarree(),
        cmap="turbo", frame_width=380,
        title=title,
    )
    plots.append(p)

pn.Row(*plots).servable()

## 10. Seasonal Climatology (2017–2024)

In [ ]:
ds_clim = ds_monthly["VHM0"].groupby("time.season").mean()

ds_clim.hvplot(
    x="longitude", y="latitude",
    groupby="season",
    geo=True, tiles="OSM", project=True,
    crs=ccrs.PlateCarree(),
    cmap="plasma", clim=(0, 5),
    frame_width=550,
    title="Seasonal mean significant wave height — VHM0 (m), 2017–2024",
)

## 11. Annual Mean Time-series for Multiple Variables

In [ ]:
ds_annual = ds_monthly.resample(time="1YE").mean()

ts_plots = []
for var, title in var_titles.items():
    if var not in ds_annual:
        continue
    ts = ds_annual[var].sel(longitude=LON_PT, latitude=LAT_PT, method="nearest")
    ts_plots.append(
        ts.hvplot.line(
            label=title, height=250, responsive=True,
            title=f"{title} — annual mean at lon={LON_PT}, lat={LAT_PT}",
            ylabel=var,
        )
    )

pn.Column(*ts_plots).servable()